# Malinois for K562 enhancer design

[Installation](../docs/installation.md)

Design 200-nt enhancers that favor K562 activity over HepG2 and SK-N-SH.

## 1. Load Malinois



In [1]:
# %pip install "bioseq-models @ git+https://github.com/UTR-CODE/bioseq-models.git@84a5075c3656127d99027c38705212a2de6e62c9"

from pathlib import Path

import torch
from bioseq_models import load_model
from MotifSift import call_motifs, generate_designs, random_seed_sequences, select_candidates

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
model = load_model("malinois", device=device, batch_size=1024)
run = Path("outputs/malinois_k562")




Malinois adds fixed flanks when it encodes DNA strings. Use the same flanks for gradient-based scoring.



In [2]:
context = model.encoder(["A" * 200]).transpose(1, 2)

def add_flanks(probabilities):
    batch = len(probabilities)
    left = context[:, :200].to(probabilities).expand(batch, -1, -1)
    right = context[:, -200:].to(probabilities).expand(batch, -1, -1)
    return torch.cat((left, probabilities, right), dim=1).transpose(1, 2)

model.soft_encoder = add_flanks

def specificity(y):
    return y["K562_log2FC"] - torch.maximum(y["HepG2_log2FC"], y["SKNSH_log2FC"])




## 2. Generate candidates



In [3]:
seeds = random_seed_sequences(200, 100, seed=42)
designs = generate_designs(
    model, length=200, initial=seeds, objective=specificity, seed=42,
    output_dir=run / "designs",
)
designs.groupby("algorithm")["score"].agg(["size", "median", "max"])


,size,median,max
algorithm,,,
adalead,100,28.846767,30.252395
basesearch,100,23.288496,25.481710
beam_search,100,27.427982,30.767609
cross_entropy_method,100,28.943088,29.598214
den,100,26.630059,28.707355
fast_seqprop,100,27.131786,29.043246
genetic_algorithm,100,33.412852,34.887352
ism_greedy,100,28.653624,31.538803
ledidi,100,24.038023,26.400880




## 3. Discover motifs



In [4]:
motifs = call_motifs(designs, model, output_dir=run / "motifs")
motifs.support.query("retained")[["motif_id", "seqlets", "parents", "sites_on_unique_DNA"]]


,motif_id,seqlets,parents,sites_on_unique_DNA
0,1-VAGATAAGG,3009,1058,5309.0
1,2-AMCAKMTGKT,544,476,1626.0
2,3-GTGASTCAC,317,258,849.0
3,4-TTCCWGGAA,55,45,545.0
5,7-CCACACCC,17,17,258.0
7,9-CTTCCCGCCA,27,27,183.0
11,14-CAGCAGCTGCA,38,38,45.0
12,15-ATCTGTTCCTG,29,27,37.0
13,16-GATAATGAC,254,248,174.0
19,28-ATCTGGTGTC,32,32,176.0




## 4. Select a panel



In [5]:
gate = float(designs.drop_duplicates("sequence").score.quantile(0.2))
selected = select_candidates(designs, motifs=motifs, top_n=10, min_score=gate, output_dir=run / "selected")
selected[["selection_rank", "sequence", "score", "present_motifs"]]


,selection_rank,sequence,score,present_motifs
0,1,GTTATCAAACCGCTGTGTCTGGTTCGTACAGATAACGCTGACATTT...,34.887352,1-VAGATAAGG;2-AMCAKMTGKT;3-GTGASTCAC;4-TTCCWGG...
449,2,GCTACACCCTTATCTTCACTTACCAGCAAAAAAACGGTTACGCCCT...,27.890123,1-VAGATAAGG
311,3,GATGTGGCGGTTAGATAAGTTAAATGTGGCTGTCAACGATAGGCCT...,28.800381,1-VAGATAAGG;7-CCACACCC;9-CTTCCCGCCA;14-CAGCAGC...
410,4,TCTTGCGCGAGGGTGCCTTATCTGTTCCTTGCAGCACTGTGGGGTT...,28.136970,1-VAGATAAGG;4-TTCCWGGAA;9-CTTCCCGCCA;15-ATCTGT...
200,5,GTTGGAGATAAGGCTTATCTCTTATCTTCCACGAGCAGCTGTTGGG...,29.043246,1-VAGATAAGG;2-AMCAKMTGKT;14-CAGCAGCTGCA
849,6,AGGAGTTATCTTATCACAAGTCTCCTGTGCACCGAAGATAAAGCTG...,23.780276,1-VAGATAAGG;3-GTGASTCAC;4-TTCCWGGAA;16-GATAATGAC
529,7,TCTTATCTTCATGCACCAGACAGCACCTGATAACCGCCAAAAAGTC...,27.073193,1-VAGATAAGG;2-AMCAKMTGKT;3-GTGASTCAC;7-CCACACCC
795,8,TTATCACGCCAAAAGCATCTGTCCGTTTTGATAAAGCGGTTACTGG...,24.203702,1-VAGATAAGG;2-AMCAKMTGKT;9-CTTCCCGCCA;16-GATAA...
170,9,GTCTTATCTAATTCTTAGAATTGACGCTCATTTCCAGGAAACCGGC...,29.207387,1-VAGATAAGG;2-AMCAKMTGKT;4-TTCCWGGAA;7-CCACACC...
171,10,TGTAGGGTGTGTCACGTTGGTGGGTGGCCCCCGCAGGAACATATGT...,29.205774,1-VAGATAAGG;2-AMCAKMTGKT;3-GTGASTCAC;14-CAGCAG...
